# 1장 Generating Your First Text — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter01/Chapter 1 - Introduction to Language Models.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter01` 노트북. 권장 인스턴스 `ml.g4dn.xlarge`(T4 16GB, 책의 Colab 기준과 동일), 스토리지 30GB 이상.

## 0. 환경 점검 — GPU·디스크·버전

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import torch, transformers
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 1. (필요 시에만) 패키지 설치

이미지에 transformers·accelerate가 이미 있으면 건너뜁니다. 위 셀에서 ImportError가 났을 때만 주석을 풀고 실행한 뒤 커널을 재시작하세요.

In [ ]:
# %pip install -q "transformers>=4.41" accelerate

## 2. 모델과 토크나이저 불러오기

책 본문은 `trust_remote_code=True`지만, 공식 저장소 노트북은 `False`로 바뀌었습니다. 최신 transformers에는 Phi-3가 내장되어 있어 `False`가 안전합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(model.dtype, f"{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU")

## 3. pipeline 만들기

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

## 4. 첫 텍스트 생성

In [ ]:
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]
output = generator(messages)
print(output[0]["generated_text"])

## 5. (학습용) 토크나이저가 실제로 만든 입력 보기 — 2장 예고

`messages`가 모델에 들어가기 전 어떤 문자열·토큰으로 바뀌는지 확인합니다.

In [ ]:
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
ids = tokenizer(prompt).input_ids
print(len(ids), 'tokens:', tokenizer.convert_ids_to_tokens(ids))

## 6. 끝나면

노트북을 저장하고 Studio에서 **Space를 Stop** 하세요. 인스턴스 요금은 멈추지만 EBS 볼륨(모델 캐시 포함)은 남아 스토리지 요금이 계속 붙습니다. 완전히 정리하려면 Space를 삭제합니다.